In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Adjust this path to wherever you've placed the `data/` folder (incl. data/beauty_full/) + this notebook in your Drive
PROJECT_DIR = os.environ.get('RECSYS_PROJECT_DIR', os.path.join('/content', 'drive', 'MyDrive', 'amazon-item-recommender'))
os.chdir(PROJECT_DIR)

# Train BPR (BGE title-embedding initialization)

Colab-oriented notebook for training BPR with BGE title-initialized item embeddings. The evaluation batch size is reduced to avoid full-sort memory pressure.

In [ ]:
!pip uninstall -y torch torchvision torchaudio transformers sentence-transformers peft accelerate
!pip install "torch==2.5.1" "torchvision==0.20.1" "torchaudio==2.5.1"
!pip install "transformers==4.46.3" "sentence-transformers==3.3.1" "accelerate<1.0"
!pip install "pandas<=2.3.2" numpy matplotlib seaborn matplotlib-venn datasets ipykernel recbole kmeans-pytorch

Found existing installation: torch 2.12.1
Uninstalling torch-2.12.1:
  Successfully uninstalled torch-2.12.1
Found existing installation: torchvision 0.27.1
Uninstalling torchvision-0.27.1:
  Successfully uninstalled torchvision-0.27.1
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: sentence-transformers 5.5.1
Uninstalling sentence-transformers-5.5.1:
  Successfully uninstalled sentence-transformers-5.5.1
Found existing installation: peft 0.19.1
Uninstalling peft-0.19.1:
  Successfully uninstalled peft-0.19.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached triton-3.1.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.3 kB)
  Using cached sympy-1.13.1-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 906.4/906.4 MB 1.7 MB/s eta

In [ ]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch" "sentence-transformers"

In [ ]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [ ]:
from typing import Any
import torch
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.general_recommender import BPR
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger
from sentence_transformers import SentenceTransformer

In [ ]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty_full"  # full data, no downsampling
DATA_DIR: str = "data"
SEED = 67
DEVICE = "cuda" # Other options: "cpu", "mps"
EMBEDDING_SIZE = 64  # matches BPR.yaml default, kept explicit so the projection step below lines up

## Create dataset

Same as `train-bpr.ipynb`, but with `title` added to the item feature columns so we can read it back out per-item below.

In [ ]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "category"],
        "item": ["item_id", "title"],
    },
    "embedding_size": EMBEDDING_SIZE,
    "epochs": 100,
    "train_batch_size": 1024,
    "eval_batch_size": 4096,  # full data is too large for a single mega-batch; use RecBole default
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None,
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
}

config: Config = Config(model="BPR", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [ ]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/usr/local/lib/python3.12/dist-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)


## Encode item titles with BGE, aligned to RecBole's internal item index

`dataset.item_feat["title"]` gives title token IDs ordered by RecBole's internal item index. `dataset.field2token_id["title"]` maps those token IDs back to title strings before BGE encoding.

In [ ]:
print('hi')

hi


In [ ]:
# Suppress httpx INFO logs from SentenceTransformer's model loading
logging.getLogger('httpx').setLevel(logging.WARNING)

title_tokens: torch.Tensor = dataset.item_feat["title"]  # (n_items,) token IDs, RecBole item-index order
id2token: dict[int, str] = {
    v: k for k, v in dataset.field2token_id["title"].items()
}
titles: list[str] = [id2token.get(tok.item(), "") for tok in title_tokens]

print(f"Encoding {len(titles):,} item titles with BGE (BAAI/bge-base-en-v1.5)...")
bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=str(config["device"]))
bge_embs = bge_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
bge_embs = torch.from_numpy(bge_embs).float()  # (n_items, 768)
print(f"BGE embeddings shape: {tuple(bge_embs.shape)}")

Encoding 250,853 item titles with BGE (BAAI/bge-base-en-v1.5)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/980 [00:00<?, ?it/s]

BGE embeddings shape: (250853, 768)


## Project BGE embeddings down to BPR's embedding size

BGE outputs 768-dim vectors; BPR here uses `EMBEDDING_SIZE=64`. A fixed random Gaussian projection maps BGE embeddings to the BPR embedding size.

In [ ]:
bge_dim = bge_embs.shape[1]

g = torch.Generator().manual_seed(SEED)
projection = torch.randn(bge_dim, EMBEDDING_SIZE, generator=g) / (bge_dim ** 0.5)

item_init_embs = bge_embs @ projection  # (n_items, EMBEDDING_SIZE)

# Zero out the padding item (index 0) so it never gets recommended
item_init_embs[0] = 0.0

print(f"Projected init embeddings shape: {tuple(item_init_embs.shape)}")

Projected init embeddings shape: (250853, 64)


## Train BPR, initialized from the projected BGE embeddings

In [ ]:
model: BPR = BPR(config, train_data.dataset).to(config["device"])

# Overwrite the Xavier-initialized item embeddings with our BGE-derived ones,
# right before training starts. Everything downstream (loss, optimizer, eval)
# is untouched — interactions are free to move these embeddings during training.
with torch.no_grad():
    model.item_embedding.weight.data.copy_(item_init_embs.to(config["device"]))

trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

/usr/local/lib/python3.12/dist-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)


In [ ]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0088
Best valid result:
  ndcg@20: 0.0088
  recall@20: 0.0170
  mrr@20: 0.0099


## Evaluate on test set

In [ ]:
len(test_data)

55594

In [ ]:
test_result: dict[str, float] = trainer.evaluate(test_data)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

/usr/local/lib/python3.12/dist-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_file, map_location=s

Test results (Overall):
  ndcg@20: 0.0083
  recall@20: 0.0147
  mrr@20: 0.0100


In [ ]:
print('hi')

hi


In [ ]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    '''
    Evaluate the model on a subset of interactions defined by `mask`.
    '''
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    print(f"\nEvaluation ({label})")
    print(f'-' * 20)
    print(f"  Interactions: {mask.sum()}")
    for metric, val in results.items():
        print(f"  {metric}: {val:.4f}")

# Map integer categories to labels
tok = dataset.field2token_id["category"]
CAT_LABELS = {tok["0"]: "warm", tok["1"]: "cold"}

uid_to_cat = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["category"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()

for cat_id, cat_label in CAT_LABELS.items():
    cat_uids = {uid for uid, c in uid_to_cat.items() if c == cat_id}
    mask = np.isin(uid_array, list(cat_uids))
    if not mask.any():
        print(f"\n  {cat_label}: no users in test set — skipping")
        continue

    evaluate_on_subset(test_data, mask, cat_label)


Evaluation (warm)
--------------------
  Interactions: 46742
  ndcg@20: 0.0076
  recall@20: 0.0127
  mrr@20: 0.0119

Evaluation (cold)
--------------------
  Interactions: 150964
  ndcg@20: 0.0084
  recall@20: 0.0150
  mrr@20: 0.0097


## Compare against plain BPR

Use the main training notebooks and README results table for current baseline comparisons.